# Pest Infestation Audio Detection Model — CNN on Mel-Spectrograms
**Tea Plantation Smart Monitoring System**

This notebook trains a CNN to classify audio recordings of tea plantation environments as:
- `infested` (pest sounds present)
- `not_infested` (ambient/clean)

**Audio pipeline** (mirrors `audio_pipeline.py`):
1. Load WAV → mono → resample 16 kHz
2. FFT bandpass filter 200–8000 Hz
3. Trim / pad to 3 seconds
4. Mel-spectrogram (128 mels, hop=512, n_fft=2048)
5. Power → dB → normalize [0, 1] → resize 128×128 → stack 3 channels

**Artifact saved:** `pest_cnn.keras`

---
### Dataset directory structure expected:
```
data/audio/
├── infested/       ← WAV files of infested environments
└── not_infested/   ← WAV files of clean environments
```
Minimum recommended: **150 WAV files per class** (300 total).


In [ ]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print(f'TensorFlow : {tf.__version__}')
print(f'librosa    : {librosa.__version__}')
print(f'GPU        : {bool(tf.config.list_physical_devices("GPU"))}')

# ─── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR      = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
AUDIO_DIR     = os.path.join(BASE_DIR, 'data', 'audio')
ARTIFACTS_DIR = os.path.join(BASE_DIR, 'tea-plantation-system', 'ml_training', 'model_artifacts')
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

CLASS_NAMES  = ['infested', 'not_infested']
NUM_CLASSES  = 2
SAMPLE_RATE  = 16000
DURATION_SEC = 3
N_MELS       = 128
HOP_LENGTH   = 512
N_FFT        = 2048
IMG_SIZE     = 128
BATCH_SIZE   = 16
EPOCHS       = 50

print(f'Audio directory: {AUDIO_DIR}')


## 1. Dataset Loading
Scan WAV files from `infested/` and `not_infested/` subdirectories.


In [ ]:
# Create placeholder directories
for cls in CLASS_NAMES:
    os.makedirs(os.path.join(AUDIO_DIR, cls), exist_ok=True)

file_paths = []
file_labels = []

for label_idx, cls in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(AUDIO_DIR, cls)
    wavs = glob.glob(os.path.join(cls_dir, '*.wav')) + \
           glob.glob(os.path.join(cls_dir, '*.WAV'))
    file_paths.extend(wavs)
    file_labels.extend([label_idx] * len(wavs))
    print(f'  {cls:<15}: {len(wavs)} WAV files found')

total_files = len(file_paths)
print(f'\nTotal WAV files: {total_files}')

if total_files == 0:
    print('\n[WARNING] No WAV files found. Add audio files to the class directories above,')
    print('          then re-run from this cell. Training will be skipped below.')
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False


## 2. Audio Preprocessing Pipeline
Mirrors `audio_pipeline.py`:
1. Load WAV → mono → resample 16 kHz
2. FFT bandpass 200–8000 Hz
3. Trim / pad to 3 s
4. Mel-spectrogram → power to dB → normalize [0,1] → resize 128×128 → stack 3 ch


In [ ]:
def fft_bandpass(signal, sr, low_hz=200, high_hz=8000):
    """Zero-out FFT bins outside [low_hz, high_hz] then IFFT."""
    spectrum = np.fft.rfft(signal)
    freqs    = np.fft.rfftfreq(len(signal), d=1.0 / sr)
    mask     = (freqs >= low_hz) & (freqs <= high_hz)
    spectrum[~mask] = 0.0
    return np.fft.irfft(spectrum, n=len(signal))


def audio_to_spectrogram(file_path,
                          sr=SAMPLE_RATE,
                          duration=DURATION_SEC,
                          n_mels=N_MELS,
                          hop_length=HOP_LENGTH,
                          n_fft=N_FFT,
                          img_size=IMG_SIZE):
    """
    Full audio pipeline:
    WAV → mono → resample → bandpass → trim/pad → mel-spec → dB → normalize → 3-ch image
    """
    # 1. Load & mono
    y, orig_sr = librosa.load(file_path, sr=None, mono=True)

    # 2. Resample to target SR
    if orig_sr != sr:
        y = librosa.resample(y, orig_sr=orig_sr, target_sr=sr)

    # 3. FFT bandpass 200–8000 Hz
    y = fft_bandpass(y, sr, low_hz=200, high_hz=8000)

    # 4. Trim silence then pad / truncate to exact duration
    y, _ = librosa.effects.trim(y, top_db=20)
    target_len = sr * duration
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode='constant')
    else:
        y = y[:target_len]

    # 5. Mel-spectrogram
    mel = librosa.feature.melspectrogram(y=y, sr=sr,
                                          n_mels=n_mels,
                                          hop_length=hop_length,
                                          n_fft=n_fft)

    # 6. Power to dB
    mel_db = librosa.power_to_db(mel, ref=np.max)

    # 7. Normalize to [0, 1]
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-6:
        mel_norm = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_norm = np.zeros_like(mel_db)

    # 8. Resize to img_size × img_size
    mel_resized = cv2.resize(mel_norm, (img_size, img_size),
                              interpolation=cv2.INTER_LINEAR)

    # 9. Stack to 3-channel image (H, W, 3)
    mel_3ch = np.stack([mel_resized, mel_resized, mel_resized], axis=-1)

    return mel_3ch.astype(np.float32)


print('Audio pipeline functions defined.')
print(f'Target output shape per sample: ({IMG_SIZE}, {IMG_SIZE}, 3)')


In [ ]:
# Process all audio files → spectrogram images
if not SKIP_TRAINING:
    print(f'Processing {total_files} audio files...')
    X_list, y_list = [], []
    errors = 0

    for idx, (fp, lbl) in enumerate(zip(file_paths, file_labels)):
        try:
            spec = audio_to_spectrogram(fp)
            X_list.append(spec)
            y_list.append(lbl)
        except Exception as e:
            print(f'  [SKIP] {os.path.basename(fp)}: {e}')
            errors += 1
        if (idx + 1) % 50 == 0:
            print(f'  Processed {idx + 1}/{total_files}')

    X = np.array(X_list, dtype=np.float32)  # (N, 128, 128, 3)
    y = np.array(y_list, dtype=np.int32)

    print(f'\nX shape : {X.shape}')
    print(f'y shape : {y.shape}')
    print(f'Errors  : {errors}')
    print(f'Class counts: {dict(zip(*np.unique(y, return_counts=True)))}')

    # Visualise a sample spectrogram from each class
    fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(10, 4))
    for cls_idx, (ax, cls_name) in enumerate(zip(axes, CLASS_NAMES)):
        sample_idx = np.where(y == cls_idx)[0]
        if len(sample_idx) > 0:
            ax.imshow(X[sample_idx[0], :, :, 0], origin='lower',
                      aspect='auto', cmap='magma')
            ax.set_title(f'{cls_name}\n(label={cls_idx})')
            ax.axis('off')
    plt.suptitle('Sample Mel-Spectrograms', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped: no audio files.')


## 3. Train / Test Split


In [ ]:
if not SKIP_TRAINING:
    y_cat = to_categorical(y, num_classes=NUM_CLASSES)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y_cat, test_size=0.2, random_state=42, stratify=y
    )
    print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
else:
    print('Skipped.')


## 4. Build CNN
```
Input(128,128,3)
→ Conv2D(32,3×3,relu) → MaxPool(2×2)
→ Conv2D(64,3×3,relu) → MaxPool(2×2)
→ Conv2D(128,3×3,relu) → MaxPool(2×2)
→ GlobalAveragePooling2D
→ Dense(128,relu) → Dropout(0.4)
→ Dense(2,softmax)
```


In [ ]:
def build_cnn(input_shape=(128, 128, 3), num_classes=2):
    inputs = keras.Input(shape=input_shape, name='spectrogram_input')

    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='pest_audio_cnn')


cnn_model = build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES)
cnn_model.summary(line_length=80)
print(f'\nTotal params: {cnn_model.count_params():,}')


## 5. Compile & Train (50 epochs + EarlyStopping)


In [ ]:
cnn_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1)
]

if not SKIP_TRAINING:
    history = cnn_model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        callbacks=callbacks_cnn,
        verbose=1
    )

    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['accuracy'],     label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title('CNN — Accuracy'); axes[0].legend(); axes[0].set_xlabel('Epoch')

    axes[1].plot(history.history['loss'],     label='Train')
    axes[1].plot(history.history['val_loss'], label='Val')
    axes[1].set_title('CNN — Loss'); axes[1].legend(); axes[1].set_xlabel('Epoch')

    plt.suptitle('Pest Audio CNN Training History', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('Skipped: no audio data.')


## 6. Evaluation — Confusion Matrix & Classification Report


In [ ]:
if not SKIP_TRAINING:
    y_pred_probs = cnn_model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    y_true = np.argmax(y_test, axis=1)

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title('Confusion Matrix — Pest Audio CNN', fontsize=13)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.show()

    print('\n=== Classification Report ===')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    test_loss, test_acc = cnn_model.evaluate(X_test, y_test, verbose=0)
    print(f'Final test accuracy: {test_acc:.4f}  |  test loss: {test_loss:.4f}')
else:
    print('Skipped: no audio data.')


## 7. Save Model


In [ ]:
if not SKIP_TRAINING:
    save_path = os.path.join(ARTIFACTS_DIR, 'pest_cnn.keras')
    cnn_model.save(save_path)
    print(f'Model saved: {save_path}')

    # Quick load verification
    loaded = keras.models.load_model(save_path)
    print(f'Verification load OK — output shape: {loaded.output_shape}')
else:
    print('No model to save: SKIP_TRAINING=True.')

print('\nPest audio model notebook complete.')
